In [81]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
import numpy as np
from scipy.stats import randint

In [41]:
train = pd.read_csv("../CleanData/full_train.csv")
test = pd.read_csv("../CleanData/full_test.csv")

In [75]:
#Function to prepare dataframes for models
def prep_data(df):
    drop_cols = ["gid", "visteam", "hometeam", "site", "date", "vruns", "hruns", "wteam", "lteam", "teamID"]
    df = df.drop(drop_cols, axis = 1)
    df_y = df["score_diff"]
    df_x = df.drop("score_diff", axis = 1)
    return [df_x, df_y]

In [49]:
df_train = prep_data(train)
df_test = prep_data(test)

In [57]:
#Hyperparameter Tuning
rf_regressor = RandomForestRegressor(random_state=42)

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None] + list(np.arange(5, 50, 5)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['auto', 'sqrt', 'log2']
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=50, 
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Fit the random search
random_search.fit(df_train[0], df_train[1])

# Get the best parameters
best_params = random_search.best_params_
print("\nBest Parameters Found:")
print(best_params)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


C:\Users\rubie\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
85 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
63 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\rubie\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\rubie\anaconda3\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "C:\Users\rubie\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\Users\rubie\anaconda3\Lib\site-packages


Best Parameters Found:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 13, 'n_estimators': 344}


In [73]:
#Cross Validation with tuned params
rf_regressor_tuned = RandomForestRegressor(**best_params, random_state=42)

#R2
r2_scores = cross_val_score(rf_regressor_tuned, df_train[0], df_train[1], cv=5, scoring='r2')
print("R² scores:", r2_scores)
print("Avg R²:", np.mean(r2_scores))

#MSE
mse_scores = cross_val_score(rf_regressor_tuned, df_train[0], df_train[1], scoring='neg_mean_squared_error')
mse_scores = -mse_scores
print("MSE scores:", mse_scores)
print("Avg MSE:", np.mean(mse_scores))

#MAE
mae_scores = cross_val_score(rf_regressor_tuned, df_train[0], df_train[1], scoring='neg_mean_absolute_error')
mae_scores = -mae_scores
print("MAE scores:", mae_scores)
print("Avg MAE:", np.mean(mae_scores))

R² scores: [0.28273201 0.27535196 0.2730987  0.28031668 0.28186596]
Avg R²: 0.2786730617405636
MSE scores: [13.51406094 13.527575   13.59644587 13.95824679 13.86488546]
Avg MSE: 13.692242812507615
MAE scores: [2.83006358 2.84675912 2.84575595 2.8860852  2.88282557]
Avg MAE: 2.858297883025687


In [83]:
#Model Evaluation
rf_regressor_tuned.fit(df_train[0], df_train[1])
y_pred = rf_regressor_tuned.predict(df_test[0])

mse = mean_squared_error(df_test[1], y_pred)
mae = mean_absolute_error(df_test[1], y_pred)
r2 = r2_score(df_test[1], y_pred)

print("MSE:", mse)
print("MAE:", mae)
print("R² Score:", r2)

MSE: 13.598701502762706
MAE: 2.8470764141512817
R² Score: 0.2829392524821679
